In [1]:
import numpy as np
import cv2
import os


os.makedirs("dataset/square",exist_ok=True)
os.makedirs("dataset/circle",exist_ok=True)

img_size=128

def generate_square():
    img =np.zeros((img_size,img_size),dtype=np.uint8)
    start = np.random.randint(10,30)
    size = np.random.randint(20,50)
    img[start:start+size,start:start+size]=255
    return img
def generate_circle():
    img = np.zeros((img_size,img_size),dtype = np.uint8)
    center = (np.random.randint(30,98),np.random.randint(30,98))
    radius = np.random.randint(10,30)
    cv2.circle(img,center,radius,255,-1)

    return img
for i in range(50):
    square_img = generate_square()
    cv2.imwrite(f"dataset/square/square_{i}.png",square_img)
    circle_img = generate_circle()
    cv2.imwrite(f"dataset/circle/circle_{i}.png",circle_img)

In [9]:
os.makedirs("dataset/Triangle",exist_ok=True)
os.makedirs("dataset/Cross",exist_ok=True)

def generatte_triangle():
    img =np.zeros((img_size,img_size),dtype=np.uint8)
    vert1 = (np.random.randint(30,98),np.random.randint(30,98))
    vert2 = (np.random.randint(20,70),np.random.randint(20,70))
    vert3 = (np.random.randint(50,100),np.random.randint(20,100))
    pts = np.array([vert1,vert2,vert3],np.int32)
    cv2.fillPoly(img,[pts],255)
    return img
def generate_cross():
    img = np.zeros((img_size,img_size),dtype=np.uint8)
    center = (np.random.randint(30,98),np.random.randint(30,98))
    size = np.random.randint(20,50)
    cv2.line(img,(center[0]-size//2,center[1]),(center[0]+size//2,center[1]),255,5)
    cv2.line(img,(center[0],center[1]-size//2),(center[0],center[1]+size//2),255,5)
    return img
for i in range(50):
    triangle_img = generatte_triangle()
    cv2.imwrite(f"dataset/Triangle/triangle_{i}.png",triangle_img)
    cross_img = generate_cross()
    cv2.imwrite(f"dataset/Cross/cross_{i}.png",cross_img)

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader,random_split
from torchvision import transforms
from PIL import Image
import glob

class ShapeDataset(Dataset):
    def __init__(self,root,transform=None):
        self.transform =transform
        self.images = []
        self.labels = []
        classes = os.listdir(root)
        self.class_to_idx = {cls_name:idx for idx,cls_name in enumerate(classes)}
        for cls in classes:
            img_paths = glob.glob(os.path.join(root,cls,"*.png"))
            self.images.extend(img_paths)
            self.labels.extend([self.class_to_idx[cls]]*len(img_paths))
    def __len__(self):
        return len(self.images)
    def __getitem__(self,idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("L")
        if self.transform:
            image = self.transform(image)
        return image,label
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])
dataset = ShapeDataset("dataset",transform=transform)
# tran_size =int(0.8*len(dataset))
print(len(dataset))


200


In [11]:
train_size = int(0.7*len(dataset))
val_size = int(0.15 *len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset,[train_size,val_size,test_size])
train_loader = DataLoader(train_dataset,batch_size = 12,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=12,shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=12,shuffle=False)


In [85]:
import torch.nn as nn
class SimpleCNN(nn.Module):
    def __init__(self,num_classes =4):
        super(SimpleCNN,self).__init__()
        self.conv1 = nn.Conv2d(1,16,3,stride =2,padding=1)
        self.conv2 = nn.Conv2d(16,32,3,stride =2 , padding =1)
        self.fc1 = nn.Linear(32*32*32,128)
        self.fc2 = nn.Linear(128,num_classes)
        self.relu = nn.ReLU()
    def forward(self,x):
        x= self.conv1(x)
        x= self.relu(x)
        x= self.conv2(x)
        x= self.relu(x)
        x= x.view(x.size(0),-1)
        x= self.fc1(x)
        x= self.relu(x)
        x= self.fc2(x)
        return x


In [86]:
model=SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
num_epochs = 10
print(model)
print(model.forward(torch.randn( 1,1, 128, 128)))

SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=4, bias=True)
  (relu): ReLU()
)
tensor([[ 0.0063, -0.0164, -0.0549,  0.0646]], grad_fn=<AddmmBackward0>)


In [24]:
for _ in range(num_epochs):
    model.train()
    for images,labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch [{_+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
    for images,labels in val_loader:
        model.eval()
        with torch.no_grad():
            outputs = model(images)
            val_loss = criterion(outputs,labels)
    print(f"Validation Loss: {val_loss.item():.4f}")

Epoch [1/10], Step [50/12], Loss: 0.0002
Validation Loss: 6.3858
Epoch [2/10], Step [50/12], Loss: 0.0003
Validation Loss: 6.4299
Epoch [3/10], Step [50/12], Loss: 0.0004
Validation Loss: 6.4806
Epoch [4/10], Step [50/12], Loss: 0.0003
Validation Loss: 6.5333
Epoch [5/10], Step [50/12], Loss: 0.0003
Validation Loss: 6.5762
Epoch [6/10], Step [50/12], Loss: 0.0002
Validation Loss: 6.6187
Epoch [7/10], Step [50/12], Loss: 0.0001
Validation Loss: 6.6682
Epoch [8/10], Step [50/12], Loss: 0.0004
Validation Loss: 6.7067
Epoch [9/10], Step [50/12], Loss: 0.0001
Validation Loss: 6.7591
Epoch [10/10], Step [50/12], Loss: 0.0003
Validation Loss: 6.7706


In [25]:
for images,labels in test_loader:
    model.eval()
    with torch.no_grad():
        outputs = model(images)
        _,predicted = torch.max(outputs.data,1)
        total = labels.size(0)
        correct = (predicted == labels).sum().item()
print(f"Test Accuracy: {100*correct/total:.2f}%")

Test Accuracy: 66.67%


In [ ]:
import torchvision.transforms as transforms
import torchvision.models as models

resnet = models.resnet18(pretrained = True)

class Resnet18(nn.Module):
    def __init__(self,num_classes =4):
        super(Resnet18,self).__init__()
        self.resnet = resnet
        # self.resnet.layer3[1].conv2= nn.Conv2d(128,64,3,stride=1,padding=1,bias=False)
        # self.resnet.conv1 =nn.Conv2d(in_channels = 1,out_channels=64,kernel_size=7,stride=2,padding=3,bias=False)
        self.resnet.avgpool = nn.MaxPool2d(4)
        # self.relu = nn.ReLU()
        # for param in resnet.parameters():
        #     param.requires_grad = False
        # resnet.fc.requires_grad = True
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features,num_classes)
    def forward(self,x):
        x=self.resnet(x)
        # return self.relu(x)
        return x
model = Resnet18()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
num_epochs = 10
# print(model)
print(model)
print(model.forward(torch.randn( 1,3, 128, 128)))

Resnet18(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_run

In [67]:
for epoch in range(num_epochs):
    model.train()
    for images,labels in train_loader:
        images = images.repeat(1,3,1,1)
        outputs = model(images)
        loss = criterion(outputs,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
    for images,labels in val_loader:
        model.eval()
        with torch.no_grad():
            images = images.repeat(1,3,1,1)
            outputs = model(images)
            val_loss = criterion(outputs,labels)
    print(f"Validation Loss: {val_loss.item():.4f}")

Epoch [1/10], Step [50/12], Loss: 0.7705
Validation Loss: 0.0001
Epoch [2/10], Step [50/12], Loss: 0.1502
Validation Loss: 0.0001
Epoch [3/10], Step [50/12], Loss: 0.2180
Validation Loss: 0.0032
Epoch [4/10], Step [50/12], Loss: 0.0169
Validation Loss: 0.0000
Epoch [5/10], Step [50/12], Loss: 0.0010
Validation Loss: 0.0029
Epoch [6/10], Step [50/12], Loss: 0.0079
Validation Loss: 0.0095
Epoch [7/10], Step [50/12], Loss: 0.0004
Validation Loss: 0.0004
Epoch [8/10], Step [50/12], Loss: 0.5784
Validation Loss: 0.0001
Epoch [9/10], Step [50/12], Loss: 0.0031
Validation Loss: 0.0000
Epoch [10/10], Step [50/12], Loss: 0.0006
Validation Loss: 0.0000


In [68]:
for images,labels in test_loader:
    model.eval()
    with torch.no_grad():
        images = images.repeat(1,3,1,1)
        outputs = model(images)
        _,predicted = torch.max(outputs.data,1)
        total = labels.size(0)
        correct = (predicted == labels).sum().item()
print(f"Test Accuracy: {100*correct/total:.2f}%")

Test Accuracy: 100.00%


In [69]:
for param in resnet.parameters():
    param.requires_grad = False
resnet.fc.requires_grad = True

resnet = models.resnet18(pretrained = True)
resnet.fc = nn.Linear(resnet.fc.in_features,4)
model = Resnet18()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
num_epochs = 10
print(model.forward(torch.randn( 1,3, 128, 128)))
print(model)

/home/ashant/Desktop/75dayplan/env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ashant/Desktop/75dayplan/env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


tensor([[-3.1121,  0.9762,  1.8569,  1.7705]], grad_fn=<AddmmBackward0>)
Resnet18(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     

In [70]:
for param in resnet.layer4.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),lr=0.001)
for epoch in range(num_epochs):
    model.train()
    for images,labels in train_loader:
        images = images.repeat(1,3,1,1)
        outputs = model(images)
        loss = criterion(outputs,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
    for images,labels in val_loader:
        model.eval()
        with torch.no_grad():
            images = images.repeat(1,3,1,1)
            outputs = model(images)
            val_loss = criterion(outputs,labels)
    print(f"Validation Loss: {val_loss.item():.4f}")

Epoch [1/10], Step [50/12], Loss: 0.0028
Validation Loss: 0.0099
Epoch [2/10], Step [50/12], Loss: 0.0011
Validation Loss: 0.0000
Epoch [3/10], Step [50/12], Loss: 0.0001
Validation Loss: 0.0000
Epoch [4/10], Step [50/12], Loss: 0.0001
Validation Loss: 0.0000
Epoch [5/10], Step [50/12], Loss: 0.0001
Validation Loss: 0.0000
Epoch [6/10], Step [50/12], Loss: 0.0029
Validation Loss: 0.0000
Epoch [7/10], Step [50/12], Loss: 0.0000
Validation Loss: 0.0001
Epoch [8/10], Step [50/12], Loss: 0.0021
Validation Loss: 0.0000
Epoch [9/10], Step [50/12], Loss: 0.0034
Validation Loss: 0.0000
Epoch [10/10], Step [50/12], Loss: 0.0009
Validation Loss: 0.0000


In [71]:
for images,labels in test_loader:
    model.eval()
    with torch.no_grad():
        images = images.repeat(1,3,1,1)
        outputs = model(images)
        _,predicted = torch.max(outputs.data,1)
        total = labels.size(0)
        correct = (predicted == labels).sum().item()
print(f"Test Accuracy: {100*correct/total:.2f}%")

Test Accuracy: 100.00%


In [72]:
import torch.nn.functional as F
class BasicBlock(nn.Module):
    def __init__(self,in_channels,out_channels,stride=1):
        super(BasicBlock,self).__init__()
        self.conv1=nn.Conv2d(in_channels,out_channels,3,stride=stride,padding=1,bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels,out_channels,3,stride=1,padding=1,bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride !=1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels,out_channels,1,stride=stride,bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self,x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(identity)
        return F.relu(out)

In [75]:
class MiniResnet(nn.Module):
    def __init__(self,num_classes =4):
        super(MiniResnet,self).__init__()
        self.conv1=nn.Conv2d(1,64,3,stride=1,padding=1,bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64,64,2,stride=1)
        self.layer2 = self._make_layer(64,128,2,stride=2)
        # self.layer3 = self._make_layer(128,256,2,stride=2)
        # self.layer4 = self._make_layer(256,512,2,stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(128,num_classes)
    def _make_layer(self,in_channels,out_channels,blocks,stride):
        layers = []
        layers.append(BasicBlock(in_channels,out_channels,stride))
        for _ in range(1,blocks):
            layers.append(BasicBlock(out_channels,out_channels))
        return nn.Sequential(*layers)
    def forward(self,x):
        x=F.relu(self.bn1(self.conv1(x)))
        x=self.layer1(x)
        x=self.layer2(x)
        # x=self.layer3(x)
        # x=self.layer4(x)
        x=self.avgpool(x)
        x=x.view(x.size(0),-1)
        x=self.fc(x)
        return x
model = MiniResnet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
num_epochs = 10
print(model)
print(model.forward(torch.randn( 1,1, 128, 128)))
        

MiniResnet(
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, aff

In [76]:
for _ in range(num_epochs):
    model.train()
    for images,labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch [{_+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
    for images,labels in val_loader:
        model.eval()
        with torch.no_grad():
            outputs = model(images)
            val_loss = criterion(outputs,labels)
    print(f"Validation Loss: {val_loss.item():.4f}")

Epoch [1/10], Step [50/12], Loss: 1.2628
Validation Loss: 1.1441
Epoch [2/10], Step [50/12], Loss: 1.1197
Validation Loss: 1.2445
Epoch [3/10], Step [50/12], Loss: 0.9998
Validation Loss: 1.8811
Epoch [4/10], Step [50/12], Loss: 0.8262
Validation Loss: 11.4362
Epoch [5/10], Step [50/12], Loss: 0.8957
Validation Loss: 5.7549
Epoch [6/10], Step [50/12], Loss: 0.5503
Validation Loss: 11.6128
Epoch [7/10], Step [50/12], Loss: 0.6798
Validation Loss: 4.2370
Epoch [8/10], Step [50/12], Loss: 0.8883
Validation Loss: 0.6019
Epoch [9/10], Step [50/12], Loss: 0.3848
Validation Loss: 1.2285
Epoch [10/10], Step [50/12], Loss: 0.3480
Validation Loss: 5.9267


In [78]:
for images,labels in test_loader:
    model.eval()
    with torch.no_grad():
        # images = images.repeat(1,3,1,1)
        outputs = model(images)
        _,predicted = torch.max(outputs.data,1)
        total = labels.size(0)
        correct = (predicted == labels).sum().item()
print(f"Test Accuracy: {100*correct/total:.2f}%")

Test Accuracy: 50.00%


"""
Mini Vision Transformer
"""

In [79]:
class PatchEmbedding(nn.Module):
    def __init__(self,img_size=128,patch_size=16,in_channels=1,embed_dim=64):
        super(PatchEmbedding,self).__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels,embed_dim,kernel_size=patch_size,stride=patch_size)
    def forward(self,x):
        x = self.proj(x) #(B,embed_dim,H/patch_size,W/patch_size)
        x = x.flatten(2) #(B,embed_dim,num_patches)
        x = x.transpose(1,2) #(B,num_patches,embed_dim)
        return x

In [80]:
class PositionalEncoding(nn.Module):
    def __init__(self,num_patches,embed_dim):
        super(PositionalEncoding,self).__init__()
        self.pos_embedding = nn.Parameter(torch.zeros(1,num_patches,embed_dim))
    def forward(self,x):
        return x + self.pos_embedding

In [81]:
class TransformerBlock(nn.Module):
    def __init__(self,embed_dim,num_heads,mlp_dim,dropout=0.1):
        super(TransformerBlock,self).__init__()
        self.attn = nn.MultiheadAttention(embed_dim,num_heads,dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim,mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim,embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
    def forward(self,x):
        attn_output,_ = self.attn(x,x,x)
        x = self.norm1(x + attn_output)
        mlp_output = self.mlp(x)
        x = self.norm2(x + mlp_output)
        return x

In [82]:
class MiniViT(nn.Module):
    def __init__(self,img_size=128,patch_size=16,in_channels=1,embed_dim=64,num_heads=4,mlp_dim=128,num_classes=4,num_blocks=4):
        super(MiniViT,self).__init__()
        self.patch_embed = PatchEmbedding(img_size,patch_size,in_channels,embed_dim)
        self.pos_embed = PositionalEncoding(self.patch_embed.num_patches,embed_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim,num_heads,mlp_dim) for _ in range(num_blocks)
        ])
        self.classifier = nn.Linear(embed_dim,num_classes)
    def forward(self,x):
        x = self.patch_embed(x)
        x = self.pos_embed(x)
        for block in self.blocks:
            x = block(x)
        x = x.mean(dim=1) # Global average pooling
        x = self.classifier(x)
        return x
model = MiniViT()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
num_epochs = 10
print(model)
print(model.forward(torch.randn( 1,1, 128, 128)))

MiniViT(
  (patch_embed): PatchEmbedding(
    (proj): Conv2d(1, 64, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_embed): PositionalEncoding()
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=64, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
  )
  (classifier): Linear(in_features=64, out_features=4, bias=True)
)
tensor([[-0.0899, -0.1758, -0.0881, -0.5455]], grad_fn=<AddmmBackward0>)


In [83]:
for _ in range(num_epochs):
    model.train()
    for images,labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch [{_+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
    for images,labels in val_loader:
        model.eval()
        with torch.no_grad():
            outputs = model(images)
            val_loss = criterion(outputs,labels)
    print(f"Validation Loss: {val_loss.item():.4f}")

Epoch [1/10], Step [50/12], Loss: 1.3204
Validation Loss: 1.3530
Epoch [2/10], Step [50/12], Loss: 1.4130
Validation Loss: 1.3235
Epoch [3/10], Step [50/12], Loss: 1.3706
Validation Loss: 1.2980
Epoch [4/10], Step [50/12], Loss: 1.3315
Validation Loss: 1.2807
Epoch [5/10], Step [50/12], Loss: 1.3950
Validation Loss: 1.2798
Epoch [6/10], Step [50/12], Loss: 1.3565
Validation Loss: 1.2748
Epoch [7/10], Step [50/12], Loss: 1.3854
Validation Loss: 1.2580
Epoch [8/10], Step [50/12], Loss: 1.3445
Validation Loss: 1.2692
Epoch [9/10], Step [50/12], Loss: 1.2964
Validation Loss: 1.2488
Epoch [10/10], Step [50/12], Loss: 1.1847
Validation Loss: 1.2567


In [84]:
for images,labels in test_loader:
    model.eval()
    with torch.no_grad():
        # images = images.repeat(1,3,1,1)
        outputs = model(images)
        _,predicted = torch.max(outputs.data,1)
        total = labels.size(0)
        correct = (predicted == labels).sum().item()
print(f"Test Accuracy: {100*correct/total:.2f}%")

Test Accuracy: 16.67%
